# Optimal Estimation Demo

This notebook demonstrates the O2 A optimal-estimation tooling with the aerosol state vector used by the retained validation case.


## What This Demonstrates

The example retrieves aerosol optical depth and aerosol layer top altitude from a simulated measurement, then displays the result through `result.plot` accessors. Each plot has its own cell.


In [1]:
import json

import zdisamar as zd
from zdisamar.inverse_method import optimal_estimation as oe
from zdisamar.inverse_method.optimal_estimation import o2a as o2a_oe

from validation.common import o2a_retrieval_baseline as oe_baseline
from validation.common.o2a_measurement_noise import measurement_from_o2a_baseline_noise

REFERENCE_PATH = zd.reference_data.path("validation/disamar_o2a_two_state_reference.json")


def load_reference() -> dict:
    with REFERENCE_PATH.open() as handle:
        return json.load(handle)


def apply_aerosol_layer(
    case: zd.O2AInput,
    *,
    top_pressure_hpa: float,
    bottom_pressure_hpa: float,
) -> None:
    case.aerosol.placement.top_pressure_hpa = top_pressure_hpa
    case.aerosol.placement.bottom_pressure_hpa = bottom_pressure_hpa
    interval_index = case.aerosol.placement.interval_index_1based
    for interval in case.atmosphere.intervals:
        if interval.index_1based == interval_index:
            interval.top_pressure_hpa = top_pressure_hpa
            interval.bottom_pressure_hpa = bottom_pressure_hpa
        elif interval.index_1based == interval_index - 1:
            interval.bottom_pressure_hpa = top_pressure_hpa
        elif interval.index_1based == interval_index + 1:
            interval.top_pressure_hpa = bottom_pressure_hpa


def case_from_reference(reference: dict, section: str) -> zd.O2AInput:
    case = zd.o2a_disamar_reference_input()
    oe_baseline.configure_case(case)
    scene = reference[section]
    case.aerosol.optical_depth_550_nm = float(scene["aerosol_optical_depth"])
    apply_aerosol_layer(
        case,
        top_pressure_hpa=float(scene["aerosol_layer_top_pressure_hpa"]),
        bottom_pressure_hpa=float(scene["aerosol_layer_bottom_pressure_hpa"]),
    )
    return case


def build_measurement(case: zd.O2AInput, reference: dict) -> oe.Measurement:
    _ = reference
    with zd.prepare(case) as prepared:
        return measurement_from_o2a_baseline_noise(prepared)


def build_state_vector(
    case: zd.O2AInput,
    reference: dict,
    profile: oe.PressureAltitudeProfile,
) -> oe.StateVector:
    prior = reference["a_priori"]
    layer_thickness_hpa = float(reference["truth"]["aerosol_layer_bottom_pressure_hpa"]) - float(
        reference["truth"]["aerosol_layer_top_pressure_hpa"]
    )
    return oe.StateVector(
        [
            oe.AerosolOpticalDepth(
                initial=float(prior["aerosol_optical_depth"]),
                prior=float(prior["aerosol_optical_depth"]),
                variance=1.0,
                lower=0.0,
            ),
            oe.AerosolLayerTopAltitude(
                initial=float(prior["aerosol_layer_top_altitude_km"]),
                prior=float(prior["aerosol_layer_top_altitude_km"]),
                variance=float(prior["aerosol_layer_top_altitude_variance_km2"]),
                pressure_thickness_hpa=layer_thickness_hpa,
                interval_index_1based=case.aerosol.placement.interval_index_1based,
                pressure_altitude_profile=profile,
            ),
        ]
    )


def run_optimal_estimation_demo() -> oe.Result:
    reference = load_reference()
    truth_case = case_from_reference(reference, "truth")
    inverse_case = case_from_reference(reference, "a_priori")
    measurement = build_measurement(truth_case, reference)
    with zd.prepare(inverse_case) as prepared:
        profile = o2a_oe.pressure_altitude_profile_from_prepared(prepared)
    state_vector = build_state_vector(inverse_case, reference, profile)
    return oe.disamar_oe(
        inverse_model=oe.O2AInverseForwardModel(inverse_case),
        measurement=measurement,
        state_vector=state_vector,
        controls=oe.RetrievalControls.from_disamar_retrieval_specs(),
    )

## Run The Retrieval

This cell runs the retrieval. The following cells display the charts directly from `result.plot`.


In [2]:
result = run_optimal_estimation_demo()
{
    "state_names": result.state_names,
    "retrieved_state": result.state.tolist(),
    "iterations": result.iterations,
    "converged": result.converged,
}

{'state_names': ('aerosol_optical_depth', 'aerosol_layer_top_altitude_km'),
 'retrieved_state': [0.30000000383540926, 1.252759684436742],
 'iterations': 4,
 'converged': True}

## Retrieval Convergence


In [3]:
result.plot.convergence()

alt.Chart(...)

## Measurement Fit


In [4]:
result.plot.measurement_fit()

alt.Chart(...)

## Final Residual


In [5]:
result.plot.residual()

alt.Chart(...)

## Final Reflectance Jacobians


In [6]:
result.plot.jacobian()

alt.Chart(...)